# Scenario Dreamer Baseline (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_baseline_colab.ipynb)

This notebook creates the **baseline experimental substrate** on Colab:
- sync the repo,
- mount Google Drive,
- bind the persistent checkpoint/environment-pack layout,
- install a lean Scenario Dreamer simulation runtime,
- run a smoke-tier baseline evaluation,
- render a demo MP4 while writing run artifacts directly to Drive.


In [ ]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Suggested defaults for a Drive-backed smoke baseline
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_RUN_SETUP=1
%env SCENARIO_DREAMER_RUN_SMOKE=1
%env SCENARIO_DREAMER_RUN_DEMO=1


In [ ]:
# Step 2: Mount Drive, clone/pin upstream Scenario Dreamer, and bind Drive-backed assets/results
import json
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))


In [ ]:
# Step 3: Install a lean runtime for Scenario Dreamer simulation on Colab
import os
import subprocess
import sys

RUN_SETUP = os.environ.get('SCENARIO_DREAMER_RUN_SETUP', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SETUP:', RUN_SETUP)
if RUN_SETUP:
    subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py', '--editable-project'], check=True)
else:
    print('Skipping runtime setup.')


In [ ]:
# Step 4: Dry-run the smoke-tier baseline and confirm missing assets before execution
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_smoke_eval.py', '--dry-run'], check=True)


In [ ]:
# Step 5: Execute the smoke-tier baseline run
import json
import os
import subprocess
import sys

RUN_SMOKE = os.environ.get('SCENARIO_DREAMER_RUN_SMOKE', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SMOKE:', RUN_SMOKE)
smoke_payload = None
if RUN_SMOKE:
    raw = subprocess.check_output([sys.executable, 'scripts/run_smoke_eval.py'], text=True)
    print(raw)
    smoke_payload = json.loads(raw)
else:
    print('Skipping smoke run.')


In [ ]:
# Step 6: Render a single-environment demo MP4
import json
import os
import subprocess
import sys

RUN_DEMO = os.environ.get('SCENARIO_DREAMER_RUN_DEMO', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_DEMO:', RUN_DEMO)
demo_payload = None
if RUN_DEMO:
    raw = subprocess.check_output([sys.executable, 'scripts/render_demo.py'], text=True)
    print(raw)
    demo_payload = json.loads(raw)
else:
    print('Skipping demo render.')


In [ ]:
# Step 7: Inspect the latest Drive-backed run artifacts and preview the first MP4 if present
import json
import os
from pathlib import Path

from IPython.display import Video, display

results_root = Path(os.environ['SCENARIO_DREAMER_RESULTS_ROOT'])
run_dirs = sorted([p for p in results_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
print('results_root:', results_root)
print('num_run_dirs:', len(run_dirs))
if not run_dirs:
    raise RuntimeError('No run directories found. Execute Step 5 or Step 6 first.')

latest = run_dirs[0]
print('latest_run_dir:', latest)
for name in ['run_manifest.json', 'metrics.json', 'config_snapshot.json', 'video_manifest.json']:
    path = latest / name
    if path.exists():
        print(f'
{name}:')
        print(path.read_text()[:4000])

movies = sorted(latest.rglob('*.mp4'))
print('movie_count:', len(movies))
if movies:
    display(Video(str(movies[0]), embed=True))
